# Credit Default Prediction with Logistic Regression

This notebook develops a small-sample binary classifier for `Defaulted`. It excludes `Applicant_ID`, keeps the source CSV read-only, fits preprocessing inside every validation fold to prevent leakage, and compares logistic regression with a prior-probability baseline.

**Evidence status:** the notebook is delivered unexecuted. Results become runtime evidence only after execution in the intended environment.

## Development plan

1. Load and validate the source data without modifying it.
2. Exclude the applicant identifier and separate predictors from the binary target.
3. Inspect dimensions, missingness, and class support without displaying applicant-level records.
4. Build a leakage-safe pipeline: median imputation, standardization, and L2-regularized logistic regression.
5. Compare it with a prior-probability dummy baseline using identical repeated stratified folds.
6. Summarize discrimination, probability quality, and threshold-based performance.
7. Aggregate repeated out-of-fold probabilities for a confusion matrix and diagnostic metrics.
8. Fit a final descriptive model on all available observations and report standardized coefficients and odds ratios.
9. Record limitations; do not save predictions, transformed data, or a serialized model automatically.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    log_loss,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("credit_risk_screening.csv")
TARGET = "Defaulted"
IDENTIFIER_COLUMNS = ["Applicant_ID"]
FEATURES = ["Annual_Income", "Debt_to_Income_Ratio"]
REQUIRED_COLUMNS = IDENTIFIER_COLUMNS + FEATURES + [TARGET]
N_SPLITS = 5
N_REPEATS = 20
CLASSIFICATION_THRESHOLD = 0.50

## Load and validate the immutable source

The CSV is only read. This notebook does not overwrite it or create a derived dataset.

In [2]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Expected source dataset at: {DATA_PATH.resolve()}")

data = pd.read_csv(DATA_PATH)

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(data.columns))
unexpected_columns = sorted(set(data.columns) - set(REQUIRED_COLUMNS))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")
if unexpected_columns:
    raise ValueError(f"Unexpected columns require review: {unexpected_columns}")
if data.empty:
    raise ValueError("The dataset contains no observations.")
if data[TARGET].isna().any():
    raise ValueError("The target contains missing values; define a handling policy before modeling.")

observed_target_values = set(data[TARGET].unique())
if not observed_target_values.issubset({0, 1}) or len(observed_target_values) != 2:
    raise ValueError(
        f"{TARGET} must contain both binary values 0 and 1; observed {sorted(observed_target_values)}"
    )

for column in FEATURES:
    converted = pd.to_numeric(data[column], errors="coerce")
    newly_missing = converted.isna() & data[column].notna()
    if newly_missing.any():
        raise ValueError(f"{column} contains non-numeric values that require review.")
    data[column] = converted

if data[IDENTIFIER_COLUMNS[0]].duplicated().any():
    raise ValueError("Applicant_ID is not unique; possible duplicate applicants require review.")

summary = pd.DataFrame(
    {
        "row_count": [len(data)],
        "feature_count": [len(FEATURES)],
        "positive_targets": [int((data[TARGET] == 1).sum())],
        "negative_targets": [int((data[TARGET] == 0).sum())],
    }
)
display(summary)
display(data[FEATURES + [TARGET]].isna().sum().rename("missing_values").to_frame())

,row_count,feature_count,positive_targets,negative_targets
0,50,2,25,25


,missing_values
Annual_Income,3
Debt_to_Income_Ratio,3
Defaulted,0


## Define predictors, target, and leakage-safe models

Median imputation and scaling are learned only from each training fold. The identifier is explicitly omitted. L2 regularization is used because the dataset is small.

In [3]:
X = data[FEATURES].copy()
y = data[TARGET].astype(int).copy()

logistic_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                solver="liblinear",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

baseline_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", DummyClassifier(strategy="prior")),
    ]
)

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)
splits = list(cv.split(X, y))
len(splits)

100

## Repeated stratified cross-validation

Each model receives identical train/test indices. Fold-level percentiles below describe empirical variation; they are not formal confidence intervals because repeated folds are statistically dependent.

In [ ]:
def evaluate_repeated_cv(estimator, model_name, X, y, splits, threshold=0.50):
    fold_rows = []
    probability_sum = np.zeros(len(y), dtype=float)
    prediction_count = np.zeros(len(y), dtype=int)

    for fold_number, (train_index, test_index) in enumerate(splits, start=1):
        fitted = clone(estimator)
        fitted.fit(X.iloc[train_index], y.iloc[train_index])
        probabilities = fitted.predict_proba(X.iloc[test_index])[:, 1]
        predictions = (probabilities >= threshold).astype(int)
        y_test = y.iloc[test_index].to_numpy()
        tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()

        fold_rows.append(
            {
                "model": model_name,
                "fold": fold_number,
                "roc_auc": roc_auc_score(y_test, probabilities),
                "log_loss": log_loss(y_test, probabilities, labels=[0, 1]),
                "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
                "sensitivity": recall_score(y_test, predictions, pos_label=1, zero_division=0),
                "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            }
        )
        probability_sum[test_index] += probabilities
        prediction_count[test_index] += 1

    if np.any(prediction_count == 0):
        raise RuntimeError("At least one observation received no out-of-fold prediction.")

    mean_oof_probability = probability_sum / prediction_count
    return pd.DataFrame(fold_rows), mean_oof_probability

logistic_folds, logistic_oof_probability = evaluate_repeated_cv(
    logistic_pipeline, "L2 logistic regression", X, y, splits, CLASSIFICATION_THRESHOLD
)
baseline_folds, baseline_oof_probability = evaluate_repeated_cv(
    baseline_pipeline, "Prior-probability baseline", X, y, splits, CLASSIFICATION_THRESHOLD
)
fold_results = pd.concat([logistic_folds, baseline_folds], ignore_index=True)

c:\Users\Bush\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Bush\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:

In [ ]:
metric_columns = ["roc_auc", "log_loss", "balanced_accuracy", "sensitivity", "specificity"]
cv_summary = (
    fold_results.groupby("model")[metric_columns]
    .agg(["mean", "std", lambda values: values.quantile(0.025), lambda values: values.quantile(0.975)])
)
cv_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in cv_summary.columns.to_flat_index()
]
cv_summary = cv_summary.rename(columns=lambda name: name.replace("<lambda_0>", "p025").replace("<lambda_1>", "p975"))
display(cv_summary.round(3))

## Aggregated out-of-fold diagnostics

Repeated predictions for each applicant are averaged before applying the fixed 0.50 threshold. The identifier remains excluded from displayed results.

In [ ]:
def aggregated_oof_metrics(y_true, probabilities, threshold=0.50):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return pd.Series(
        {
            "roc_auc": roc_auc_score(y_true, probabilities),
            "log_loss": log_loss(y_true, probabilities, labels=[0, 1]),
            "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
            "sensitivity": recall_score(y_true, predictions, pos_label=1, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp,
        }
    )

oof_comparison = pd.DataFrame(
    {
        "L2 logistic regression": aggregated_oof_metrics(y, logistic_oof_probability),
        "Prior-probability baseline": aggregated_oof_metrics(y, baseline_oof_probability),
    }
).T
display(oof_comparison.round(3))

## Final descriptive fit and coefficient interpretation

This fit uses all observations only after cross-validation. Coefficients operate on standardized, median-imputed predictors. Their exponentials are odds ratios per one training-sample standard deviation increase, conditional on the other predictor. With this sample size, treat them as exploratory rather than stable causal effects.

In [ ]:
final_model = clone(logistic_pipeline).fit(X, y)
coefficients = final_model.named_steps["classifier"].coef_[0]
coefficient_table = pd.DataFrame(
    {
        "feature": FEATURES,
        "standardized_log_odds_coefficient": coefficients,
        "odds_ratio_per_1_sd": np.exp(coefficients),
    }
)
display(coefficient_table.round(3))
print(f"Standardized intercept: {final_model.named_steps['classifier'].intercept_[0]:.3f}")

## Interpretation and limitations checklist

- Require the logistic model to outperform the dummy baseline, especially on ROC-AUC and log loss, before claiming predictive value.
- Do not interpret cross-validation fold percentiles as formal confidence intervals.
- The 0.50 classification threshold is a neutral starting point, not a business-optimized cutoff. Changing it requires explicit cost or risk criteria.
- Fifty observations and two predictors provide only preliminary evidence; external validation is unavailable.
- Median imputation assumes missingness can be handled without adding a missingness indicator. Revisit this with more data or domain evidence.
- Association does not establish causation or fairness. The available columns are insufficient for a meaningful subgroup fairness assessment.
- Do not deploy or make consequential lending decisions from this model without substantially more representative data, domain review, calibration analysis, and governance controls.
- No output is written automatically. Saving a model or derived results should be a separate, explicitly reviewed action.